# RTAA: Full Adversarial Benchmark on HSI Classifiers

This notebook trains **5 HSI classifiers** from scratch and evaluates them against **5 adversarial attacks** on **3 benchmark datasets**.

| Models | Attacks | Datasets |
|--------|---------|----------|
| HybridSN (patchwise) | FGSM | PaviaU (103 bands, 9 classes) |
| HybridSN (pixelwise) | I-FGSM | Indian Pines (200 bands, 16 classes) |
| SpectralFormer (patchwise) | PGD | Salinas (204 bands, 16 classes) |
| S3ANet | SS-FGSM | |
| SACNet | RTAA | |

All results (OA, AA, Kappa, per-class accuracy, SAM, SID, physical consistency, ASR) are exported to a formatted Excel workbook.

**Kaggle version.** Same methodology and implementation as the Colab notebook — only the environment setup (Section 0) has been adapted to Kaggle's filesystem: Google Drive mounting is replaced with `/kaggle/input` (for your uploaded `.mat` files) and `/kaggle/working` (for checkpoints/results, which Kaggle persists as notebook output). No modeling, attack, or evaluation code was changed.


## 0. Set Paths (Kaggle)


In [ ]:
# On Kaggle, uploaded data lives (read-only) under /kaggle/input/<dataset-name>/
# and all writable output should go under /kaggle/working/.
#
# Before running this notebook:
# 1. Attach a Kaggle Dataset with your .mat files (paviaU.mat, etc.) — set KAGGLE_DATASET_NAME.
# 2. Attach a Kaggle Dataset with your pre-trained checkpoints — set KAGGLE_CHECKPOINT_NAME.
#    (Upload your RTAA_checkpoints folder as a Kaggle Dataset named 'rtaa-checkpoints')
#
# The notebook will load checkpoints from the input dataset (read-only).
# Any newly trained models will be saved to /kaggle/working/ (writable).

import os

KAGGLE_DATASET_NAME = 's3anet-data'        # <-- your .mat data dataset slug
KAGGLE_CHECKPOINT_NAME = 'rtaa-checkpoints'  # <-- your checkpoints dataset slug

DATA_DIR = f'/kaggle/input/{KAGGLE_DATASET_NAME}'
CHECKPOINT_INPUT_DIR = f'/kaggle/input/{KAGGLE_CHECKPOINT_NAME}'  # read-only, pre-trained
CHECKPOINT_DIR = '/kaggle/working/RTAA_checkpoints'               # writable, for new saves
OUTPUT_DIR = '/kaggle/working/RTAA_results'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_checkpoint(filename):
    """Look for a checkpoint file: first in the uploaded input dataset, then in working dir."""
    input_path = os.path.join(CHECKPOINT_INPUT_DIR, filename)
    if os.path.exists(input_path):
        return input_path
    working_path = os.path.join(CHECKPOINT_DIR, filename)
    if os.path.exists(working_path):
        return working_path
    return None

if not os.path.isdir(DATA_DIR):
    print(f'WARNING: {DATA_DIR} not found. Attach your data dataset as an input '
          f'(top-right "+ Add Input" in Kaggle), then update KAGGLE_DATASET_NAME above.')

if not os.path.isdir(CHECKPOINT_INPUT_DIR):
    print(f'NOTE: {CHECKPOINT_INPUT_DIR} not found. Models will be trained from scratch.')
    print(f'      To skip training, upload your RTAA_checkpoints folder as a Kaggle Dataset.')
else:
    ckpt_files = [f for f in os.listdir(CHECKPOINT_INPUT_DIR) if f.endswith('.pt')]
    print(f'Found {len(ckpt_files)} checkpoint(s) in {CHECKPOINT_INPUT_DIR}:')
    for f in sorted(ckpt_files):
        print(f'  {f}')

print(f'\nData dir: {DATA_DIR}')
print(f'Checkpoint input: {CHECKPOINT_INPUT_DIR}')
print(f'Checkpoint output: {CHECKPOINT_DIR}')
print(f'Results: {OUTPUT_DIR}')


## 1. Environment Setup

In [ ]:
# Clone RTAA repo from GitHub
!git clone -b RTAA https://github.com/SRUJANPATEL3669/S3ANET.git /kaggle/working/RTAA 2>/dev/null || echo 'RTAA already cloned'

# Clone external model repositories
!git clone https://github.com/YichuXu/S3ANet.git /kaggle/working/S3ANet 2>/dev/null || echo 'S3ANet already cloned'
!git clone https://github.com/YonghaoXu/SACNet.git /kaggle/working/SACNet 2>/dev/null || echo 'SACNet already cloned'
!git clone https://github.com/danfenghong/IEEE_TGRS_SpectralFormer.git /kaggle/working/SpectralFormer 2>/dev/null || echo 'SpectralFormer already cloned'

# Set environment variables for repo paths
os.environ['S3ANET_REPO_DIR'] = '/kaggle/working/S3ANet'
os.environ['SACNET_REPO_DIR'] = '/kaggle/working/SACNet'
os.environ['SPECTRALFORMER_REPO_DIR'] = '/kaggle/working/SpectralFormer'
os.environ['RTAA_DATA_DIR'] = DATA_DIR

# Install dependencies
!pip install -q scikit-image openpyxl h5py matplotlib

# Add RTAA src to Python path so 'rtaa' package is importable
import sys
sys.path.insert(0, '/kaggle/working/RTAA/src')

print('Environment setup complete.')


## 2. Prepare Data & Dataset Splits

In [ ]:
import numpy as np
import glob

# Dataset configurations
DATASETS = {
    'PaviaU': {'n_bands': 103, 'n_classes': 9,
               'data_file': 'paviaU.mat', 'data_key': 'paviaU',
               'gt_file': 'paviaU_gt.mat', 'gt_key': 'paviaU_gt'},
    'IndianPines': {'n_bands': 200, 'n_classes': 16,
                    'data_file': 'indian_pines_corrected.mat', 'data_key': 'indian_pines_corrected',
                    'gt_file': 'indian_pines_gt.mat', 'gt_key': 'indian_pines_gt'},
    'Salinas': {'n_bands': 204, 'n_classes': 16,
                'data_file': 'Salinas_corrected.mat', 'data_key': 'salinas_corrected',
                'gt_file': 'Salinas_gt.mat', 'gt_key': 'salinas_gt'},
}

# List all .mat files available in Drive
mat_files = glob.glob(os.path.join(DATA_DIR, '*.mat'))
print(f'Found {len(mat_files)} .mat files in {DATA_DIR}:')
for f in sorted(mat_files):
    print(f'  {os.path.basename(f)}')

# Symlink ALL .mat files directly into each repo's Data/ directory
for repo in ['/kaggle/working/S3ANet', '/kaggle/working/SACNet']:
    data_dir = os.path.join(repo, 'Data')
    os.makedirs(data_dir, exist_ok=True)
    for mat_file in mat_files:
        fname = os.path.basename(mat_file)
        dst = os.path.join(data_dir, fname)
        if not os.path.exists(dst):
            os.symlink(mat_file, dst)
            print(f'  Linked {fname} -> {repo}/Data/')

print('\nData files linked.')


In [ ]:
from scipy.io import loadmat

def generate_splits(data_dir, ds_name, ds_info, train_samples=300, seed=0):
    """Generate train/test splits (X.npy, Y.npy, train_array.npy, test_array.npy)
    matching the format expected by S3ANet/SACNet training scripts.
    """
    out_dir = os.path.join(data_dir, ds_name)
    os.makedirs(out_dir, exist_ok=True)

    # Check if splits already exist
    if os.path.exists(os.path.join(out_dir, 'X.npy')):
        print(f'  {ds_name}: splits already exist, skipping.')
        return True

    # Load .mat files
    data_path = os.path.join(data_dir, ds_info['data_file'])
    gt_path = os.path.join(data_dir, ds_info['gt_file'])
    if not os.path.exists(data_path) or not os.path.exists(gt_path):
        print(f'  {ds_name}: .mat files not found, skipping.')
        print(f'    Expected: {data_path}')
        print(f'    Expected: {gt_path}')
        return False

    mat = loadmat(data_path)
    cube = mat[ds_info['data_key']].astype(np.float32)  # (H, W, n_bands)
    mat_gt = loadmat(gt_path)
    labels = mat_gt[ds_info['gt_key']].astype(np.int64)  # (H, W)

    h, w = labels.shape
    n_bands = cube.shape[-1]

    # Normalize per-band to [0, 1]
    X = cube.reshape(-1, n_bands)
    X_min = X.min(axis=0, keepdims=True)
    X_max = X.max(axis=0, keepdims=True)
    X = (X - X_min) / (X_max - X_min + 1e-12)
    X = X.reshape(n_bands, h, w).astype(np.float32)  # (n_bands, H, W) for whole-scene models

    Y = labels.reshape(-1)  # (H*W,)

    # Stratified sampling: train_samples per class
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    for cls in np.unique(Y):
        if cls == 0:  # skip background
            continue
        cls_idx = np.where(Y == cls)[0]
        rng.shuffle(cls_idx)
        n_train = min(train_samples, len(cls_idx) // 2)
        train_idx.extend(cls_idx[:n_train].tolist())
        test_idx.extend(cls_idx[n_train:].tolist())

    train_array = np.array(train_idx)
    test_array = np.array(test_idx)

    np.save(os.path.join(out_dir, 'X.npy'), X)
    np.save(os.path.join(out_dir, 'Y.npy'), Y)
    np.save(os.path.join(out_dir, 'train_array.npy'), train_array)
    np.save(os.path.join(out_dir, 'test_array.npy'), test_array)
    print(f'  {ds_name}: splits generated ({len(train_idx)} train, {len(test_idx)} test)')
    return True

# Generate splits for both repos
for repo_name, repo_path in [('S3ANet', '/kaggle/working/S3ANet'), ('SACNet', '/kaggle/working/SACNet')]:
    print(f'\n--- {repo_name} ---')
    data_dir = os.path.join(repo_path, 'Data')
    for ds_name, ds_info in DATASETS.items():
        generate_splits(data_dir, ds_name, ds_info)

print('\nDataset split generation complete.')


## 3. Core Imports & Utilities

In [ ]:
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.io import loadmat
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, TensorDataset, Subset

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Data loading utility ──
def load_mat_dataset(ds_name):
    """Load HSI cube and labels from .mat files."""
    info = DATASETS[ds_name]
    mat_data = loadmat(os.path.join(DATA_DIR, info['data_file']))
    cube = mat_data[info['data_key']].astype(np.float32)
    mat_gt = loadmat(os.path.join(DATA_DIR, info['gt_file']))
    labels = mat_gt[info['gt_key']].astype(np.int64)
    return cube, labels

def normalize_reflectance(cube):
    """Scale cube to [0, 1]."""
    return cube / (cube.max() + 1e-12)

def per_band_normalize(data):
    """Per-band min-max normalize (N, n_bands) to [0, 1]."""
    lo = data.min(axis=0, keepdims=True)
    hi = data.max(axis=0, keepdims=True)
    return (data - lo) / (hi - lo + 1e-12)

def stratified_split(labels_1d, train_fraction=0.7, seed=0):
    """Stratified train/test split on 1D label array."""
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for cls in np.unique(labels_1d):
        cls_idx = np.where(labels_1d == cls)[0]
        rng.shuffle(cls_idx)
        n_train = max(1, int(len(cls_idx) * train_fraction))
        train_idx.extend(cls_idx[:n_train].tolist())
        test_idx.extend(cls_idx[n_train:].tolist())
    return train_idx, test_idx

# ── Metrics ──
from sklearn.metrics import cohen_kappa_score

def compute_metrics(clean_preds, adv_preds, labels, clean_spectra, adv_spectra, n_classes):
    """Compute all evaluation metrics."""
    # Classification metrics on adversarial predictions
    oa = (adv_preds == labels).mean() * 100.0
    per_class = []
    for c in range(n_classes):
        mask = (labels == c)
        if mask.sum() > 0:
            per_class.append((adv_preds[mask] == c).mean() * 100.0)
        else:
            per_class.append(0.0)
    valid = [a for c, a in enumerate(per_class) if (labels == c).sum() > 0]
    aa = np.mean(valid) if valid else 0.0
    kappa = cohen_kappa_score(labels, adv_preds)

    # Attack success rate
    correct_mask = (clean_preds == labels)
    if correct_mask.sum() > 0:
        asr = (adv_preds[correct_mask] != labels[correct_mask]).sum() / correct_mask.sum() * 100.0
    else:
        asr = 0.0

    # Spectral distortion (SAM, SID)
    if clean_spectra is not None and adv_spectra is not None:
        c_flat = clean_spectra.reshape(-1, clean_spectra.shape[-1])
        a_flat = adv_spectra.reshape(-1, adv_spectra.shape[-1])
        dot = (c_flat * a_flat).sum(axis=1)
        norm_c = np.linalg.norm(c_flat, axis=1)
        norm_a = np.linalg.norm(a_flat, axis=1)
        cos_theta = np.clip(dot / (norm_c * norm_a + 1e-12), -1.0, 1.0)
        sam = float(np.mean(np.arccos(cos_theta) * 180.0 / np.pi))

        p = c_flat / (c_flat.sum(axis=1, keepdims=True) + 1e-12)
        q = a_flat / (a_flat.sum(axis=1, keepdims=True) + 1e-12)
        sid_pq = np.nansum(p * np.log(np.clip(p / (q + 1e-12), 1e-12, None)), axis=1)
        sid_qp = np.nansum(q * np.log(np.clip(q / (p + 1e-12), 1e-12, None)), axis=1)
        sid = float(np.mean(sid_pq + sid_qp))
    else:
        sam, sid = 0.0, 0.0

    # Physical consistency (simplified PCA-based)
    if clean_spectra is not None and adv_spectra is not None:
        c_t = torch.from_numpy(c_flat).float()
        a_t = torch.from_numpy(a_flat).float()
        mean_c = c_t.mean(dim=0, keepdim=True)
        centered = c_t - mean_c
        _, _, V = torch.pca_lowrank(centered, q=min(5, c_flat.shape[1]))
        centered_adv = a_t - mean_c
        recon = (centered_adv @ V @ V.T) + mean_c
        dot_pr = (a_t * recon).sum(dim=1)
        norm_a_t = torch.norm(a_t, dim=1)
        norm_r_t = torch.norm(recon, dim=1)
        cos_pr = torch.clamp(dot_pr / (norm_a_t * norm_r_t + 1e-12), -1, 1)
        angles_pr = torch.acos(cos_pr) * 180.0 / np.pi
        phys_consist = float((angles_pr < 3.0).float().mean().item() * 100.0)
    else:
        phys_consist = 100.0

    return {
        'OA': oa, 'AA': float(aa), 'Kappa': kappa,
        'class_accs': per_class, 'ASR': float(asr),
        'SAM': sam, 'SID': sid, 'phys_consistency': phys_consist,
    }

print('Utilities loaded.')


## 4. Train Models
### 4.1 HybridSN (Patchwise — 3D-2D CNN)

In [ ]:
from rtaa.models.hybridsn import HybridSN
from rtaa.data.hsi_dataset import HSIPatchDataset, apply_pca, pad_cube, build_patch_index

def train_hybridsn_patchwise(cube, labels, n_classes, patch_size=25,
                              pca_components=30, n_epochs=50, batch_size=64,
                              lr=1e-3, seed=0):
    """Train HybridSN patchwise on a given dataset."""
    torch.manual_seed(seed)
    ds = HSIPatchDataset(cube, labels, patch_size=patch_size,
                         pca_components=pca_components)
    entry_labels = [e.label for e in ds.index]
    train_idx, test_idx = stratified_split(entry_labels, 0.7, seed)
    train_loader = DataLoader(Subset(ds, train_idx), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(Subset(ds, test_idx), batch_size=batch_size, shuffle=False)

    model = HybridSN(n_bands=cube.shape[-1], n_classes=n_classes,
                     patch_size=patch_size, pca_components=pca_components).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    tic = time.time()
    for epoch in range(n_epochs):
        model.train()
        correct, total = 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
        if (epoch + 1) % 10 == 0:
            model.eval()
            tc, tt = 0, 0
            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device), y.to(device)
                    tc += (model(x).argmax(1) == y).sum().item()
                    tt += x.size(0)
            print(f'  epoch {epoch+1}/{n_epochs} train_acc={correct/total:.4f} test_acc={tc/tt:.4f}')
    train_time = time.time() - tic
    return model, train_time, test_idx

# Train on all datasets
hybridsn_models = {}
hybridsn_train_times = {}
hybridsn_test_indices = {}

for ds_name, ds_info in DATASETS.items():
    print(f'\n=== Training HybridSN-Patchwise on {ds_name} ===')
    ckpt_name = f'hybridsn_patch_{ds_name.lower()}.pt'
    ckpt_path = find_checkpoint(ckpt_name)
    cube, labels = load_mat_dataset(ds_name)
    if ckpt_path is not None:
        print(f'  Loading checkpoint from {ckpt_path}')
        model = HybridSN(n_bands=ds_info['n_bands'], n_classes=ds_info['n_classes'],
                         patch_size=25, pca_components=30).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        hybridsn_models[ds_name] = model
        hybridsn_train_times[ds_name] = 0.0
    else:
        model, t_time, test_idx = train_hybridsn_patchwise(
            cube, labels, ds_info['n_classes'])
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt_name))
        hybridsn_models[ds_name] = model
        hybridsn_train_times[ds_name] = t_time
    model.eval()
    print(f'  Done. Train time: {hybridsn_train_times[ds_name]:.1f}s')


### 4.2 HybridSN (Pixelwise — 1D Spectral CNN)

In [ ]:
from rtaa.models.hybridsn_pixelwise import HybridSNPixelwise

def train_hybridsn_pixelwise(cube, labels, n_classes, n_epochs=50,
                              batch_size=128, lr=1e-3, seed=0):
    """Train pixel-level 1D CNN on individual spectra."""
    torch.manual_seed(seed)
    h, w, n_bands = cube.shape
    cube_norm = normalize_reflectance(cube)
    flat = cube_norm.reshape(-1, n_bands)
    flat_labels = labels.reshape(-1)
    mask = flat_labels != 0
    spectra = flat[mask]
    lbls = flat_labels[mask] - 1  # 0-indexed

    train_idx, test_idx = stratified_split(lbls, 0.7, seed)
    train_x = torch.from_numpy(spectra[train_idx]).float().unsqueeze(1)  # (N, 1, n_bands)
    train_y = torch.from_numpy(lbls[train_idx]).long()
    test_x = torch.from_numpy(spectra[test_idx]).float().unsqueeze(1)
    test_y = torch.from_numpy(lbls[test_idx]).long()

    train_loader = DataLoader(TensorDataset(train_x, train_y),
                              batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(test_x, test_y),
                             batch_size=batch_size, shuffle=False)

    model = HybridSNPixelwise(n_bands=n_bands, n_classes=n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    tic = time.time()
    for epoch in range(n_epochs):
        model.train()
        correct, total = 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
        if (epoch + 1) % 10 == 0:
            model.eval()
            tc, tt = 0, 0
            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device), y.to(device)
                    tc += (model(x).argmax(1) == y).sum().item()
                    tt += x.size(0)
            print(f'  epoch {epoch+1}/{n_epochs} train_acc={correct/total:.4f} test_acc={tc/tt:.4f}')
    train_time = time.time() - tic
    return model, train_time

hybridsn_pw_models = {}
hybridsn_pw_train_times = {}

for ds_name, ds_info in DATASETS.items():
    print(f'\n=== Training HybridSN-Pixelwise on {ds_name} ===')
    ckpt_name = f'hybridsn_pixel_{ds_name.lower()}.pt'
    ckpt_path = find_checkpoint(ckpt_name)
    cube, labels = load_mat_dataset(ds_name)
    if ckpt_path is not None:
        print(f'  Loading checkpoint from {ckpt_path}')
        model = HybridSNPixelwise(n_bands=ds_info['n_bands'],
                                   n_classes=ds_info['n_classes']).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        hybridsn_pw_models[ds_name] = model
        hybridsn_pw_train_times[ds_name] = 0.0
    else:
        model, t_time = train_hybridsn_pixelwise(
            cube, labels, ds_info['n_classes'])
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt_name))
        hybridsn_pw_models[ds_name] = model
        hybridsn_pw_train_times[ds_name] = t_time
    model.eval()
    print(f'  Done. Train time: {hybridsn_pw_train_times[ds_name]:.1f}s')


### 4.3 SpectralFormer (Patchwise / CAF)

In [ ]:
from rtaa.models.spectralformer import SPECTRALFORMER_REPO_DIR, _VARIANT_CONFIGS, gain_neighborhood_band

if SPECTRALFORMER_REPO_DIR not in sys.path:
    sys.path.insert(0, SPECTRALFORMER_REPO_DIR)
from vit_pytorch import ViT  # type: ignore

def train_spectralformer_patchwise(cube, labels, n_classes, n_bands,
                                    patch_size=7, band_patch=3,
                                    n_epochs=300, batch_size=64,
                                    lr=5e-4, seed=0):
    """Train SpectralFormer patchwise (CAF) from scratch."""
    torch.manual_seed(seed)
    h, w, nb = cube.shape
    margin = patch_size // 2

    # Per-band normalize
    flat = cube.reshape(-1, nb)
    flat_norm = per_band_normalize(flat)
    cube_norm = flat_norm.reshape(h, w, nb)
    padded = np.pad(cube_norm, ((margin, margin), (margin, margin), (0, 0)), mode='reflect')

    # Extract patches for labeled pixels
    rows, cols = np.where(labels != 0)
    lbls = labels[rows, cols] - 1
    patches = np.empty((len(rows), patch_size, patch_size, nb), dtype=np.float32)
    for i, (r, c) in enumerate(zip(rows, cols)):
        rp, cp = r + margin, c + margin
        patches[i] = padded[rp-margin:rp+margin+1, cp-margin:cp+margin+1, :]

    # Apply group-wise spectral embedding
    patches_t = torch.from_numpy(patches).float()
    gnb_data = gain_neighborhood_band(patches_t, band_patch)  # (N, n_bands, p*p*bp)

    train_idx, test_idx = stratified_split(lbls, 0.7, seed)
    train_x = gnb_data[train_idx]
    train_y = torch.from_numpy(lbls[train_idx]).long()
    test_x = gnb_data[test_idx]
    test_y = torch.from_numpy(lbls[test_idx]).long()

    train_loader = DataLoader(TensorDataset(train_x, train_y),
                              batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(test_x, test_y),
                             batch_size=batch_size, shuffle=False)

    config = dict(_VARIANT_CONFIGS['patchwise'])
    model = ViT(num_patches=n_bands, num_classes=n_classes, **config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=n_epochs//10, gamma=0.9)
    loss_fn = nn.CrossEntropyLoss()

    tic = time.time()
    for epoch in range(n_epochs):
        model.train()
        correct, total = 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
        scheduler.step()
        if (epoch + 1) % 50 == 0:
            model.eval()
            tc, tt = 0, 0
            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device), y.to(device)
                    tc += (model(x).argmax(1) == y).sum().item()
                    tt += x.size(0)
            print(f'  epoch {epoch+1}/{n_epochs} train_acc={correct/total:.4f} test_acc={tc/tt:.4f}')
    train_time = time.time() - tic
    return model, train_time

sf_models = {}
sf_train_times = {}

for ds_name, ds_info in DATASETS.items():
    print(f'\n=== Training SpectralFormer-Patchwise on {ds_name} ===')
    ckpt_name = f'spectralformer_patch_{ds_name.lower()}.pt'
    ckpt_path = find_checkpoint(ckpt_name)
    cube, labels = load_mat_dataset(ds_name)
    if ckpt_path is not None:
        print(f'  Loading checkpoint from {ckpt_path}')
        config = dict(_VARIANT_CONFIGS['patchwise'])
        model = ViT(num_patches=ds_info['n_bands'], num_classes=ds_info['n_classes'], **config).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        sf_models[ds_name] = model
        sf_train_times[ds_name] = 0.0
    else:
        model, t_time = train_spectralformer_patchwise(
            cube, labels, ds_info['n_classes'], ds_info['n_bands'])
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt_name))
        sf_models[ds_name] = model
        sf_train_times[ds_name] = t_time
    model.eval()
    print(f'  Done. Train time: {sf_train_times[ds_name]:.1f}s')


### 4.4 S3ANet

In [ ]:
from rtaa.models.s3anet import S3ANET_REPO_DIR, load_s3anet

if S3ANET_REPO_DIR not in sys.path:
    sys.path.insert(0, S3ANET_REPO_DIR)

def train_s3anet(ds_name, n_classes, n_bands, bins=(1,2,3,6),
                 n_epochs=1000, lr=5e-4, weight_decay=5e-5):
    """Train S3ANet using the upstream repo's data pipeline."""
    from Model_S3ANet import S3ANet as S3ANetModel, CrossEntropy2d, adjust_learning_rate  # type: ignore
    from HyperTools import CalAccuracy  # type: ignore

    data_dir = f'{S3ANET_REPO_DIR}/Data/{ds_name}/'
    X = np.load(data_dir + 'X.npy')
    _, h, w = X.shape
    Y = np.load(data_dir + 'Y.npy')
    train_array = np.load(data_dir + 'train_array.npy')
    test_array = np.load(data_dir + 'test_array.npy')

    Y_train = np.full(Y.shape, 255)
    Y_train[train_array] = Y[train_array]

    model = S3ANetModel(num_features=n_bands, num_classes=n_classes,
                        bins=list(bins)).to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                weight_decay=weight_decay)
    images = torch.from_numpy(X.reshape(1, n_bands, h, w)).float().to(device)
    labels_t = torch.from_numpy(Y_train.reshape(1, h, w)).long().to(device)
    criterion = CrossEntropy2d().to(device)

    tic = time.time()
    for epoch in range(n_epochs):
        adjust_learning_rate(optimizer, lr, epoch, n_epochs)
        optimizer.zero_grad()
        loss = criterion(model(images), labels_t)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 200 == 0:
            print(f'  epoch {epoch+1}/{n_epochs} loss={loss.item():.4f}')
    train_time = time.time() - tic

    model.eval()
    with torch.no_grad():
        predict = model(images).argmax(1).squeeze(0).cpu().numpy().reshape(-1)
    OA, kappa, _ = CalAccuracy(predict[test_array], Y[test_array])
    print(f'  OA={OA*100:.2f}% Kappa={kappa*100:.2f}%')
    return model, train_time

s3anet_models = {}
s3anet_train_times = {}

for ds_name, ds_info in DATASETS.items():
    print(f'\n=== Training S3ANet on {ds_name} ===')
    ckpt_name = f's3anet_{ds_name.lower()}.pt'
    ckpt_path = find_checkpoint(ckpt_name)
    if ckpt_path is not None:
        print(f'  Loading checkpoint from {ckpt_path}')
        from Model_S3ANet import S3ANet as S3ANetModel  # type: ignore
        model = S3ANetModel(num_features=ds_info['n_bands'],
                            num_classes=ds_info['n_classes'],
                            bins=[1,2,3,6]).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        s3anet_models[ds_name] = model
        s3anet_train_times[ds_name] = 0.0
    else:
        try:
            model, t_time = train_s3anet(ds_name, ds_info['n_classes'],
                                         ds_info['n_bands'])
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt_name))
            s3anet_models[ds_name] = model
            s3anet_train_times[ds_name] = t_time
        except Exception as e:
            print(f'  S3ANet training failed for {ds_name}: {e}')
            s3anet_models[ds_name] = None
            s3anet_train_times[ds_name] = 0.0
    if s3anet_models[ds_name] is not None:
        s3anet_models[ds_name].eval()
    print(f'  Done. Train time: {s3anet_train_times[ds_name]:.1f}s')


### 4.5 SACNet

In [ ]:
from rtaa.models.sacnet import SACNET_REPO_DIR, load_sacnet

if SACNET_REPO_DIR not in sys.path:
    sys.path.insert(0, SACNET_REPO_DIR)

def train_sacnet(ds_name, n_classes, n_bands, n_epochs=1000,
                 lr=5e-4, weight_decay=5e-5):
    """Train SACNet using the upstream repo's data pipeline."""
    from Models import SACNet as SACNetModel, CrossEntropy2d, adjust_learning_rate  # type: ignore
    from HyperTools import CalAccuracy  # type: ignore

    data_dir = f'{SACNET_REPO_DIR}/Data/{ds_name}/'
    X = np.load(data_dir + 'X.npy')
    _, h, w = X.shape
    Y = np.load(data_dir + 'Y.npy')
    train_array = np.load(data_dir + 'train_array.npy')
    test_array = np.load(data_dir + 'test_array.npy')

    Y_train = np.full(Y.shape, 255)
    Y_train[train_array] = Y[train_array]

    model = SACNetModel(num_features=n_bands, num_classes=n_classes).to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                weight_decay=weight_decay)
    images = torch.from_numpy(X.reshape(1, n_bands, h, w)).float().to(device)
    labels_t = torch.from_numpy(Y_train.reshape(1, h, w)).long().to(device)
    criterion = CrossEntropy2d().to(device)

    tic = time.time()
    for epoch in range(n_epochs):
        adjust_learning_rate(optimizer, lr, epoch, n_epochs)
        optimizer.zero_grad()
        loss = criterion(model(images), labels_t)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 200 == 0:
            print(f'  epoch {epoch+1}/{n_epochs} loss={loss.item():.4f}')
    train_time = time.time() - tic

    model.eval()
    with torch.no_grad():
        predict = model(images).argmax(1).squeeze(0).cpu().numpy().reshape(-1)
    OA, kappa, _ = CalAccuracy(predict[test_array], Y[test_array])
    print(f'  OA={OA*100:.2f}% Kappa={kappa*100:.2f}%')
    return model, train_time

sacnet_models = {}
sacnet_train_times = {}

for ds_name, ds_info in DATASETS.items():
    print(f'\n=== Training SACNet on {ds_name} ===')
    ckpt_name = f'sacnet_{ds_name.lower()}.pt'
    ckpt_path = find_checkpoint(ckpt_name)
    if ckpt_path is not None:
        print(f'  Loading checkpoint from {ckpt_path}')
        from Models import SACNet as SACNetModel  # type: ignore
        model = SACNetModel(num_features=ds_info['n_bands'],
                            num_classes=ds_info['n_classes']).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        sacnet_models[ds_name] = model
        sacnet_train_times[ds_name] = 0.0
    else:
        try:
            model, t_time = train_sacnet(ds_name, ds_info['n_classes'],
                                         ds_info['n_bands'])
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt_name))
            sacnet_models[ds_name] = model
            sacnet_train_times[ds_name] = t_time
        except Exception as e:
            print(f'  SACNet training failed for {ds_name}: {e}')
            sacnet_models[ds_name] = None
            sacnet_train_times[ds_name] = 0.0
    if sacnet_models[ds_name] is not None:
        sacnet_models[ds_name].eval()
    print(f'  Done. Train time: {sacnet_train_times[ds_name]:.1f}s')


## 5. Train RTM Surrogate

In [ ]:
from rtaa.rtm.surrogate import RTMSurrogate
from rtaa.rtm.placeholder_physics import generate_synthetic_rtm_pairs

rtm_surrogates = {}

for ds_name, ds_info in DATASETS.items():
    n_bands = ds_info['n_bands']
    ckpt_name = f'rtm_surrogate_{n_bands}bands.pt'
    ckpt_path = find_checkpoint(ckpt_name)
    if ckpt_path is not None:
        print(f'Loading RTM surrogate for {n_bands} bands from {ckpt_path}...')
        surrogate = RTMSurrogate.from_pretrained(ckpt_path, n_bands=n_bands).to(device)
    else:
        print(f'Training RTM surrogate for {n_bands} bands...')
        surrogate = RTMSurrogate(n_bands=n_bands).to(device)
        # Generate synthetic training data
        atm_states, t_atm_targets, l_path_targets = generate_synthetic_rtm_pairs(
            n_samples=5000, n_bands=n_bands, seed=42)
        atm_t = torch.from_numpy(atm_states).float().to(device)
        t_atm_t = torch.from_numpy(t_atm_targets).float().to(device)
        l_path_t = torch.from_numpy(l_path_targets).float().to(device)

        optimizer = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
        for epoch in range(200):
            t_pred, l_pred = surrogate(atm_t)
            loss = F.mse_loss(t_pred, t_atm_t) + F.mse_loss(l_pred, l_path_t)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 50 == 0:
                print(f'  RTM epoch {epoch+1}/200 loss={loss.item():.6f}')
        torch.save(surrogate.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt_name))
    surrogate.eval()
    rtm_surrogates[ds_name] = surrogate
    print(f'  RTM surrogate ready for {ds_name} ({n_bands} bands)')


## 6. Define Attack Functions

All attack functions are defined inline. Includes both **patch-based** attacks (for HybridSN, SpectralFormer) and **whole-scene** attacks (for S3ANet, SACNet).

In [ ]:
# ─────────────────────────────────────────────
# Patch-based attacks (from rtaa.attacks.baselines)
# ─────────────────────────────────────────────
from rtaa.attacks.baselines import (
    fgsm_attack, pgd_attack, ifgsm_attack, ssfgsm_attack,
    ssfgsm_attack_full_scene
)
from rtaa.attacks.rtaa_attack import RTAAAttack, DifferentiablePCA, PhysicalViabilityWeights
from rtaa.attacks.spectralformer_attack import SpectralFormerRTAAAttack
from rtaa.attacks.sacnet_attack import SACNetRTAAAttack
from rtaa.rtm.mismatch import AtmosphericMismatchConfig

IGNORE_LABEL = 255

# ─────────────────────────────────────────────
# Whole-scene attacks (inline for S3ANet/SACNet)
# ─────────────────────────────────────────────

def fgsm_attack_full_scene(model, scene, labels, epsilon=0.05,
                           ignore_label=255):
    """FGSM on whole-scene models. scene: (1, n_bands, H, W)."""
    scene_adv = scene.clone().detach().requires_grad_(True)
    logits = model(scene_adv)
    # Masked cross entropy
    n, c, h, w = logits.shape
    mask = labels != ignore_label
    logits_flat = logits.permute(0,2,3,1)[mask.view(1,h,w)].view(-1, c)
    labels_flat = labels[mask]
    loss = F.cross_entropy(logits_flat, labels_flat)
    grad = torch.autograd.grad(loss, scene_adv)[0]
    return torch.clamp(scene_adv + epsilon * grad.sign(), 0.0, 1.0).detach()

def pgd_attack_full_scene(model, scene, labels, epsilon=0.05,
                          step_size=0.01, n_steps=20,
                          random_start=True, ignore_label=255):
    """PGD on whole-scene models. scene: (1, n_bands, H, W)."""
    original = scene.clone().detach()
    n_classes_local = model(original).shape[1]
    _, _, h, w = original.shape

    if random_start:
        delta = torch.empty_like(original).uniform_(-epsilon, epsilon)
    else:
        delta = torch.zeros_like(original)

    for _ in range(n_steps):
        adv = (original + delta).detach().requires_grad_(True)
        logits = model(adv)
        mask = labels != ignore_label
        logits_flat = logits.permute(0,2,3,1)[mask.view(1,h,w)].view(-1, logits.shape[1])
        labels_flat = labels[mask]
        loss = F.cross_entropy(logits_flat, labels_flat)
        grad = torch.autograd.grad(loss, adv)[0]
        delta = delta + step_size * grad.sign()
        delta = torch.clamp(delta, -epsilon, epsilon)

    return torch.clamp(original + delta, 0.0, 1.0).detach()

def ifgsm_attack_full_scene(model, scene, labels, epsilon=0.05,
                            n_steps=20, ignore_label=255):
    """I-FGSM on whole-scene models."""
    return pgd_attack_full_scene(model, scene, labels, epsilon,
                                 epsilon/n_steps, n_steps,
                                 random_start=False,
                                 ignore_label=ignore_label)

EPSILON = 0.05
N_STEPS = 20
print('Attack functions defined.')


## 7. Run All Attacks & Collect Results

In [ ]:
all_results = []
N_ATTACK_SAMPLES = 300  # Number of test samples per attack evaluation

for ds_name, ds_info in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'  DATASET: {ds_name} ({ds_info["n_bands"]} bands, {ds_info["n_classes"]} classes)')
    print(f'{"="*60}')

    cube, labels_map = load_mat_dataset(ds_name)
    cube_norm = normalize_reflectance(cube)
    h, w, n_bands = cube.shape
    n_classes = ds_info['n_classes']
    surrogate = rtm_surrogates[ds_name]
    solar = (torch.rand(n_bands) * 0.5 + 0.75).to(device)
    atm_state_default = torch.tensor([[0.05, 0.8, 15.0]]).to(device)

    # ── Helper: Get test pixel spectra ──
    rows_all, cols_all = np.nonzero(labels_map != 0)
    all_lbls = labels_map[rows_all, cols_all] - 1  # 0-indexed
    _, test_idx = stratified_split(all_lbls, 0.7, 0)
    rng = np.random.default_rng(42)
    sel = rng.choice(len(test_idx), min(N_ATTACK_SAMPLES, len(test_idx)), replace=False)
    test_sel = [test_idx[s] for s in sel]
    test_rows = rows_all[test_sel]
    test_cols = cols_all[test_sel]
    test_labels_np = all_lbls[test_sel]

    # ──────────────────────────────────────────
    # Model dispatch: for each model, run all attacks
    # ──────────────────────────────────────────
    MODEL_CONFIGS = [
        ('HybridSN-Patchwise', 'patch'),
        ('HybridSN-Pixelwise', 'pixel'),
        ('SpectralFormer-Patchwise', 'sf_patch'),
        ('S3ANet', 'whole_scene'),
        ('SACNet', 'whole_scene'),
    ]
    ATTACKS = ['Clean', 'FGSM', 'I-FGSM', 'PGD', 'SS-FGSM', 'RTAA']

    for model_name, model_type in MODEL_CONFIGS:
        print(f'\n--- {model_name} ---')

        # Get the model
        if model_name == 'HybridSN-Patchwise':
            model = hybridsn_models.get(ds_name)
            train_time = hybridsn_train_times.get(ds_name, 0)
        elif model_name == 'HybridSN-Pixelwise':
            model = hybridsn_pw_models.get(ds_name)
            train_time = hybridsn_pw_train_times.get(ds_name, 0)
        elif model_name == 'SpectralFormer-Patchwise':
            model = sf_models.get(ds_name)
            train_time = sf_train_times.get(ds_name, 0)
        elif model_name == 'S3ANet':
            model = s3anet_models.get(ds_name)
            train_time = s3anet_train_times.get(ds_name, 0)
        elif model_name == 'SACNet':
            model = sacnet_models.get(ds_name)
            train_time = sacnet_train_times.get(ds_name, 0)
        else:
            continue

        if model is None:
            print(f'  Model not available, skipping.')
            continue
        model.eval()

        # ── Prepare inputs based on model type ──
        if model_type == 'patch':
            # Extract PCA patches for HybridSN-Patchwise
            from rtaa.data.hsi_dataset import apply_pca, pad_cube
            patch_size = 25
            pca_components = 30
            margin = patch_size // 2
            pca_cube = apply_pca(cube, pca_components)
            padded = pad_cube(pca_cube, margin)

            patches = np.empty((len(test_rows), pca_components, patch_size, patch_size),
                               dtype=np.float32)
            for i, (r, c) in enumerate(zip(test_rows, test_cols)):
                rp, cp = r + margin, c + margin
                p = padded[rp-margin:rp+margin+1, cp-margin:cp+margin+1, :]
                patches[i] = np.transpose(p, (2, 0, 1))
            clean_input = torch.from_numpy(patches).float().unsqueeze(1).to(device)
            test_labels = torch.from_numpy(test_labels_np).long().to(device)

            # Clean predictions
            with torch.no_grad():
                clean_preds = model(clean_input).argmax(1).cpu().numpy()

        elif model_type == 'pixel':
            # Pixelwise: individual spectra
            spectra = cube_norm[test_rows, test_cols, :]  # (N, n_bands)
            clean_input = torch.from_numpy(spectra).float().unsqueeze(1).to(device)  # (N, 1, n_bands)
            test_labels = torch.from_numpy(test_labels_np).long().to(device)

            with torch.no_grad():
                clean_preds = model(clean_input).argmax(1).cpu().numpy()

        elif model_type == 'sf_patch':
            # SpectralFormer patchwise: 7x7 patches with gain_neighborhood_band
            sf_patch_size = 7
            sf_band_patch = 3
            margin = sf_patch_size // 2
            flat = cube.reshape(-1, n_bands)
            flat_norm = per_band_normalize(flat)
            cube_pbn = flat_norm.reshape(h, w, n_bands)
            padded = np.pad(cube_pbn, ((margin,margin),(margin,margin),(0,0)), mode='reflect')

            sf_patches = np.empty((len(test_rows), sf_patch_size, sf_patch_size, n_bands),
                                  dtype=np.float32)
            for i, (r, c) in enumerate(zip(test_rows, test_cols)):
                rp, cp = r + margin, c + margin
                sf_patches[i] = padded[rp-margin:rp+margin+1, cp-margin:cp+margin+1, :]

            sf_patches_t = torch.from_numpy(sf_patches).float().to(device)
            clean_input = gain_neighborhood_band(sf_patches_t, sf_band_patch)
            test_labels = torch.from_numpy(test_labels_np).long().to(device)

            with torch.no_grad():
                clean_preds = model(clean_input).argmax(1).cpu().numpy()

        elif model_type == 'whole_scene':
            # Whole-scene: (1, n_bands, H, W)
            # Per-band min-max normalize (matching S3ANet/SACNet training)
            flat_for_scene = cube.reshape(-1, n_bands)
            lo = flat_for_scene.min(axis=0, keepdims=True)
            hi = flat_for_scene.max(axis=0, keepdims=True)
            flat_norm_scene = (flat_for_scene - lo) / (hi - lo + 1e-12)
            scene_norm = flat_norm_scene.reshape(h, w, n_bands).transpose(2, 0, 1)  # (n_bands, H, W)
            clean_input = torch.from_numpy(scene_norm).float().unsqueeze(0).to(device)
            # Create label map for masked loss (255 for non-test pixels)
            ws_labels = np.full(h * w, 255, dtype=np.int64)
            ws_test_flat_idx = test_rows * w + test_cols
            ws_labels[ws_test_flat_idx] = test_labels_np
            ws_labels_2d = torch.from_numpy(ws_labels.reshape(h, w)).long().to(device)
            test_labels = torch.from_numpy(test_labels_np).long().to(device)

            with torch.no_grad():
                ws_logits = model(clean_input)  # (1, n_classes, H, W)
                ws_preds_map = ws_logits.argmax(1).squeeze(0).cpu().numpy().reshape(-1)
                clean_preds = ws_preds_map[ws_test_flat_idx]

        # ── Run each attack ──
        for attack_name in ATTACKS:
            print(f'  Attack: {attack_name}', end=' ... ')
            tic = time.time()

            try:
                if attack_name == 'Clean':
                    adv_preds = clean_preds.copy()
                    clean_spec_np = None
                    adv_spec_np = None

                elif model_type in ('patch', 'pixel', 'sf_patch'):
                    # ── Patch/pixel attacks ──
                    if attack_name == 'FGSM':
                        adv = fgsm_attack(model, clean_input, test_labels, epsilon=EPSILON)
                    elif attack_name == 'I-FGSM':
                        adv = ifgsm_attack(model, clean_input, test_labels,
                                           epsilon=EPSILON, n_steps=N_STEPS)
                    elif attack_name == 'PGD':
                        adv = pgd_attack(model, clean_input, test_labels,
                                         epsilon=EPSILON, step_size=EPSILON/5,
                                         n_steps=N_STEPS)
                    elif attack_name == 'SS-FGSM':
                        adv = ssfgsm_attack(model, clean_input, test_labels,
                                            epsilon=EPSILON, n_steps=N_STEPS)
                    elif attack_name == 'RTAA':
                        # RTAA attack needs special handling per model type
                        if model_type == 'sf_patch':
                            rtaa = SpectralFormerRTAAAttack(
                                surrogate=surrogate, solar_irradiance=solar,
                                epsilon=EPSILON, step_size=EPSILON/5,
                                n_steps=N_STEPS, band_patch=sf_band_patch,
                                mismatch_config=AtmosphericMismatchConfig())
                            atm = atm_state_default.expand(len(test_rows), -1)
                            adv, _ = rtaa.generate(model, sf_patches_t, test_labels, atm)
                            adv = gain_neighborhood_band(adv, sf_band_patch)
                        elif model_type == 'pixel':
                            # For pixelwise, use SpectralFormerRTAAAttack with band_patch=None
                            rtaa = SpectralFormerRTAAAttack(
                                surrogate=surrogate, solar_irradiance=solar,
                                epsilon=EPSILON, step_size=EPSILON/5,
                                n_steps=N_STEPS, band_patch=None,
                                mismatch_config=AtmosphericMismatchConfig())
                            pixel_spectra = cube_norm[test_rows, test_cols, :]
                            pixel_t = torch.from_numpy(pixel_spectra).float().to(device)
                            atm = atm_state_default.expand(len(test_rows), -1)
                            # Wrap model to transpose (B, n_bands, 1) -> (B, 1, n_bands)
                            class _PixelwiseWrapper(torch.nn.Module):
                                def __init__(self, m): super().__init__(); self.m = m
                                def forward(self, x): return self.m(x.permute(0, 2, 1))
                            adv_spectra, _ = rtaa.generate(_PixelwiseWrapper(model), pixel_t, test_labels, atm)
                            # Wrap back to (N, 1, n_bands) for the model
                            adv = adv_spectra.unsqueeze(1)
                        else:
                            # HybridSN patchwise — use RTAAAttack
                            # Build PCA projector
                            from sklearn.decomposition import PCA as SkPCA
                            flat_cube = cube_norm.reshape(-1, n_bands)
                            pca_obj = SkPCA(n_components=pca_components, whiten=True)
                            pca_obj.fit(flat_cube)
                            pca_proj = DifferentiablePCA(
                                mean=torch.from_numpy(pca_obj.mean_.astype(np.float32)).to(device),
                                components=torch.from_numpy(pca_obj.components_.astype(np.float32)).to(device),
                                whiten_scale=torch.from_numpy(np.sqrt(pca_obj.explained_variance_).astype(np.float32)).to(device)
                            )
                            # Extract raw reflectance patches
                            raw_patches = np.empty((len(test_rows), patch_size, patch_size, n_bands), dtype=np.float32)
                            padded_raw = np.pad(cube_norm, ((margin,margin),(margin,margin),(0,0)), mode='reflect')
                            for i, (r, c) in enumerate(zip(test_rows, test_cols)):
                                rp, cp = r + margin, c + margin
                                raw_patches[i] = padded_raw[rp-margin:rp+margin+1, cp-margin:cp+margin+1, :]
                            raw_patches_t = torch.from_numpy(raw_patches).float().to(device)
                            atm = atm_state_default.expand(len(test_rows), -1)
                            rtaa = RTAAAttack(
                                surrogate=surrogate, solar_irradiance=solar,
                                epsilon=EPSILON, step_size=EPSILON/5,
                                n_steps=N_STEPS,
                                mismatch_config=AtmosphericMismatchConfig())
                            adv_raw, _ = rtaa.generate(
                                model, pca_proj, raw_patches_t,
                                clean_input, test_labels, atm)
                            # Project back through PCA for the model
                            adv_pca = pca_proj(adv_raw).permute(0,3,1,2).unsqueeze(1)
                            adv = adv_pca

                    with torch.no_grad():
                        adv_preds = model(adv).argmax(1).cpu().numpy()
                    clean_spec_np = clean_input.cpu().numpy().reshape(len(test_rows), -1)
                    adv_spec_np = adv.cpu().numpy().reshape(len(test_rows), -1)

                else:  # whole_scene
                    if attack_name == 'FGSM':
                        adv_scene = fgsm_attack_full_scene(
                            model, clean_input, ws_labels_2d, epsilon=EPSILON)
                    elif attack_name == 'I-FGSM':
                        adv_scene = ifgsm_attack_full_scene(
                            model, clean_input, ws_labels_2d, epsilon=EPSILON,
                            n_steps=N_STEPS)
                    elif attack_name == 'PGD':
                        adv_scene = pgd_attack_full_scene(
                            model, clean_input, ws_labels_2d, epsilon=EPSILON,
                            step_size=EPSILON/5, n_steps=N_STEPS)
                    elif attack_name == 'SS-FGSM':
                        adv_scene = ssfgsm_attack_full_scene(
                            model, clean_input, ws_labels_2d, epsilon=EPSILON,
                            n_steps=N_STEPS)
                    elif attack_name == 'RTAA':
                        scene_for_rtaa = cube_norm.transpose(2,0,1)  # (n_bands,H,W)
                        scene_t = torch.from_numpy(scene_for_rtaa).float().to(device)
                        rtaa = SACNetRTAAAttack(
                            surrogate=surrogate, solar_irradiance=solar,
                            epsilon=EPSILON, step_size=EPSILON/5,
                            n_steps=N_STEPS,
                            mismatch_config=AtmosphericMismatchConfig())
                        adv_scene_3d, _ = rtaa.generate(
                            model, scene_t, ws_labels_2d, atm_state_default)
                        adv_scene = adv_scene_3d.unsqueeze(0)

                    with torch.no_grad():
                        adv_logits = model(adv_scene)
                        adv_preds_map = adv_logits.argmax(1).squeeze(0).cpu().numpy().reshape(-1)
                        adv_preds = adv_preds_map[ws_test_flat_idx]

                    clean_spec_np = clean_input.squeeze(0).cpu().numpy().transpose(1,2,0).reshape(-1, n_bands)[ws_test_flat_idx]
                    adv_spec_np = adv_scene.squeeze(0).cpu().numpy().transpose(1,2,0).reshape(-1, n_bands)[ws_test_flat_idx]

                attack_time = time.time() - tic

                metrics = compute_metrics(
                    clean_preds, adv_preds, test_labels_np,
                    clean_spec_np, adv_spec_np, n_classes)

                result = {
                    'dataset': ds_name,
                    'model': model_name,
                    'attack': attack_name,
                    'train_time': train_time,
                    'test_time': attack_time,
                    'total_time': train_time + attack_time,
                    **metrics,
                }
                all_results.append(result)
                print(f'OA={metrics["OA"]:.1f}% ASR={metrics["ASR"]:.1f}% '
                      f'SAM={metrics["SAM"]:.3f} ({attack_time:.1f}s)')

            except Exception as e:
                print(f'FAILED: {e}')
                all_results.append({
                    'dataset': ds_name, 'model': model_name,
                    'attack': attack_name, 'OA': 0, 'AA': 0,
                    'Kappa': 0, 'class_accs': [0]*n_classes,
                    'ASR': 0, 'SAM': 0, 'SID': 0,
                    'phys_consistency': 0, 'train_time': 0,
                    'test_time': 0, 'total_time': 0,
                })

print(f'\n\nTotal results collected: {len(all_results)}')


## 8. Save Results to Excel

In [ ]:
from rtaa.eval.excel_writer import write_benchmark_results

excel_path = os.path.join(OUTPUT_DIR, 'adversarial_benchmark_results.xlsx')
write_benchmark_results(all_results, excel_path)
print(f'Results saved to: {excel_path}')

# Also save raw JSON for backup
json_path = os.path.join(OUTPUT_DIR, 'adversarial_benchmark_results.json')
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f'JSON backup saved to: {json_path}')


## 9. Results Summary

In [ ]:
# Print summary table
print(f'{"Dataset":<15} {"Model":<25} {"Attack":<10} {"OA (%)":<10} {"ASR (%)":<10} {"SAM":<10} {"Kappa":<10}')
print('=' * 90)
for r in all_results:
    print(f'{r["dataset"]:<15} {r["model"]:<25} {r["attack"]:<10} '
          f'{r.get("OA", 0):>7.2f}   {r.get("ASR", 0):>7.2f}   '
          f'{r.get("SAM", 0):>7.4f}   {r.get("Kappa", 0):>7.4f}')


## 10. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# ── ASR Heatmap per dataset ──
for ds_name in DATASETS:
    ds_results = [r for r in all_results if r['dataset'] == ds_name and r['attack'] != 'Clean']
    if not ds_results:
        continue

    models = sorted(set(r['model'] for r in ds_results))
    attacks = [a for a in ['FGSM', 'I-FGSM', 'PGD', 'SS-FGSM', 'RTAA']
               if any(r['attack'] == a for r in ds_results)]

    asr_matrix = np.zeros((len(models), len(attacks)))
    for i, m in enumerate(models):
        for j, a in enumerate(attacks):
            match = [r for r in ds_results if r['model'] == m and r['attack'] == a]
            if match:
                asr_matrix[i, j] = match[0].get('ASR', 0)

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(asr_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(range(len(attacks)))
    ax.set_xticklabels(attacks, rotation=45, ha='right')
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(models)
    ax.set_title(f'Attack Success Rate (%) — {ds_name}')

    for i in range(len(models)):
        for j in range(len(attacks)):
            ax.text(j, i, f'{asr_matrix[i,j]:.1f}',
                    ha='center', va='center', fontsize=10, fontweight='bold')

    plt.colorbar(im, label='ASR (%)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'asr_heatmap_{ds_name.lower()}.png'), dpi=150)
    plt.show()

print('Visualizations saved to', OUTPUT_DIR)
